This notebook analyzes the results of individual attacks. 

It produces: 
   - A table displaying the average perplexity score for each attack in the dataset
   - A table showing the metrics for each attack against the base configuration of the pipeline
   - A table showing the best injection strategy for trigger-based attacks

In [4]:
# This line is needed to be able to import functions from the pr
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
from src.datasets.base_dataset import Dataset
import pandas as pd
import random
from statistics import mean, stdev
from tqdm import tqdm
from src.pipelines.evaluate.utilities import compute_perplexities
from IPython.display import display, HTML
from plot_results import format_experiment_data, load_statistics

In [5]:
### Here are the hyperparameters of this analysis

# Where the results are stored
results_data_root = "../data/results"

# The dataset name
dataset_name = "nqopen_small"

# Retrieval focused attacks
retrieval_attacks = ['IDEM','PAT','asc-natural-noreg','asc-aggressive-noreg','asc-aggressive-reg']
end_2_end_attacks = ['Phantom-corruption','PoisonRAG-hotflip','PoisonRAG-LM-targeted']
baselines = ['query+','query+answer','seo-documents-writer']
unoptimized = ['unoptimized']
valid_attacks_list = retrieval_attacks + end_2_end_attacks + baselines +unoptimized


# Optimization algorithms producing smooth documents
smooth_attacks = ['IDEM','query+','seo-documents-writer','query+answer','PoisonRAG-LM-targeted']

# Optimization algorithms not smooth documents
not_smooth_attacks = ['Phantom-corruption','Phantom-dos','PoisonRAG-hotflip','PAT','asc-natural-noreg','asc-aggressive-noreg','asc-aggressive-reg','Rag-n-roll']

## ASR statistics

In [6]:

# Path to the root of the benchmark results
root_results_path = f'{results_data_root}/{dataset_name}/default'

assert os.path.exists(root_results_path), f"The path {root_results_path} does not exist. Please make sure that you set correctly the data_root and dataset_name variables."

# Check if a data/ folder already exists
if not os.path.exists(f"{root_results_path}/data"):
    
    # If the data folder does not exis, the benchmark results are still in raw format and we should process them before the analysis
    format_experiment_data(root_results_path)

# Load all statistics in a single dataframe
df_results = load_statistics(root_results_path)

# Convert 'Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt' to numeric
df_results['Ben'] = pd.to_numeric(df_results['Ben'])
df_results['Mal'] = pd.to_numeric(df_results['Mal'])
df_results['Ben&mal'] = pd.to_numeric(df_results['Ben&mal'])
df_results['Hallucination'] = pd.to_numeric(df_results['Hallucination'])
df_results['Avg # mal docs in prompt'] = pd.to_numeric(df_results['Avg # mal docs in prompt'])

# Set Injection strategy to null if nan
df_results.loc[df_results['Injection strategy']=='nan','Injection strategy'] = ''

# Remove the experiments that are not valid
df_results = df_results[df_results['Optimization'].isin(valid_attacks_list)]

In [7]:

# Round to 3 decimal places 'Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt'
df_results[['Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt']] = df_results[['Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt']].round(3)
print(df_results['Optimization'].unique())

['asc-aggressive-reg' 'unoptimized' 'query+' 'seo-documents-writer'
 'PoisonRAG-hotflip' 'IDEM' 'asc-natural-noreg' 'asc-aggressive-noreg'
 'PoisonRAG-LM-targeted' 'query+answer' 'Phantom-corruption' 'PAT']


In [8]:
# Add column OptFamily to specify the family of the optimization
df_results['OptFamily'] = 'unoptimized'
df_results.loc[df_results['Optimization'].isin(retrieval_attacks), 'OptFamily'] = 'retrieval'
df_results.loc[df_results['Optimization'].isin(end_2_end_attacks), 'OptFamily'] = 'retrieval+LLM'
df_results.loc[df_results['Optimization'].isin(baselines), 'OptFamily'] = 'baseline'

# Add column 'ChangeType' to specify if the attack is smooth or not
df_results['ChangeType'] = 'unoptimized'
df_results.loc[df_results['Optimization'].isin(smooth_attacks), 'ChangeType'] = 'smooth'
df_results.loc[df_results['Optimization'].isin(not_smooth_attacks), 'ChangeType'] = 'Not smooth'

# Remove columns ['Parameter','Value']
df_results = df_results.drop(columns=['Parameter','Value'])

# Sort by 'Attack' adn 'Injection strategy'
df_results = df_results.sort_values(by=['Optimization','Injection strategy'])

df_results.loc[df_results['Injection strategy'] == 'proximity','Injection strategy'] = 'Answer'
df_results.loc[df_results['Injection strategy'] == 'prefix','Injection strategy'] = 'Document'

display(HTML(df_results.to_html(index=False)))

Optimization,Injection strategy,Ben,Mal,Ben&mal,Hallucination,Avg # mal docs in prompt,OptFamily,ChangeType
IDEM,Document,0.467,0.127,0.227,0.180,1.780,retrieval,smooth
IDEM,Answer,0.487,0.200,0.147,0.167,2.100,retrieval,smooth
PAT,Document,0.553,0.100,0.193,0.153,1.660,retrieval,Not smooth
PAT,Answer,0.633,0.120,0.113,0.133,1.640,retrieval,Not smooth
Phantom-corruption,,0.240,0.700,0.000,0.060,0.933,retrieval+LLM,Not smooth
PoisonRAG-LM-targeted,,0.433,0.313,0.200,0.053,1.000,retrieval+LLM,smooth
PoisonRAG-hotflip,,0.480,0.267,0.193,0.060,0.987,retrieval+LLM,Not smooth
asc-aggressive-noreg,Document,0.580,0.107,0.153,0.160,1.607,retrieval,Not smooth
asc-aggressive-noreg,Answer,0.613,0.113,0.147,0.127,1.680,retrieval,Not smooth
asc-aggressive-reg,Document,0.627,0.107,0.120,0.147,1.613,retrieval,Not smooth


In [9]:
# Find the most effective optimization strategy for the trigger-based optimizations: PAT, IDEM, asc-aggressive, asc-natural, query+
df_results_injection_attacks = df_results.loc[df_results['Optimization'].isin(['unoptimized','PAT','IDEM','asc-aggressive-noreg','asc-natural-noreg','asc-aggressive-reg','query+'])]
df_results_injection_attacks.loc[df_results_injection_attacks['Optimization'] == 'unoptimized', 'Injection strategy'] = 'unoptimized'

# Compute the mean statistics for each injection strategy
df_results_injection_attacks_1 = df_results_injection_attacks[['Injection strategy','Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt']].groupby(['Injection strategy']).mean().round(3)
display(HTML(df_results_injection_attacks_1.to_html(index=True)))

,Ben,Mal,Ben&mal,Hallucination,Avg # mal docs in prompt
Injection strategy,,,,,
Answer,0.594,0.152,0.124,0.129,1.814
Document,0.549,0.120,0.176,0.156,1.673
unoptimized,0.547,0.100,0.200,0.153,1.613


The following analysis divides the attacks in two groups: retrieval, retrieval+LLM, baseline depending on the components of the pipeline that they focus on. 

In [10]:
# group by df_results_attacks by 'ChangeType'
df_results_attacks_grouped = df_results[['OptFamily','Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt']].groupby(['OptFamily']).mean().round(3)
display(HTML(df_results_attacks_grouped.to_html(index=True)))

,Ben,Mal,Ben&mal,Hallucination,Avg # mal docs in prompt
OptFamily,,,,,
baseline,0.524,0.207,0.167,0.103,1.510
retrieval,0.583,0.124,0.145,0.148,1.694
retrieval+LLM,0.384,0.427,0.131,0.058,0.973
unoptimized,0.547,0.100,0.200,0.153,1.613


The following analysis divides the attacks in two groups: smooth and not smooth depending on the type of change they apply to the document. 

In [11]:
# group by df_results_attacks by 'ChangeType'
df_results_attacks_grouped = df_results[['ChangeType','Ben','Mal','Ben&mal','Hallucination','Avg # mal docs in prompt']].groupby(['ChangeType']).mean().round(3)
display(HTML(df_results_attacks_grouped.to_html(index=True)))

,Ben,Mal,Ben&mal,Hallucination,Avg # mal docs in prompt
ChangeType,,,,,
Not smooth,0.560,0.188,0.127,0.125,1.498
smooth,0.497,0.210,0.177,0.116,1.560
unoptimized,0.547,0.100,0.200,0.153,1.613
